<a href="https://colab.research.google.com/github/sw030701-ai/motor-control-optimization/blob/main/experiments/02_pid_baseline_tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02 · Baseline PID Tuning Experiment
### DC Motor Open-Loop Validation → Sequential Manual PID Tuning → Baseline Record

---

### Overview

이 notebook은 `docs/`에 정리한 v1 설계 기준에 맞춰 **Conventional PID Baseline**을 실제 simulation으로 만든다.

핵심 실행 흐름은 다음과 같다.

```text
Nominal Motor Parameters
      ↓
Open-Loop Validation
      ↓
Reachable Reference Selection
      ↓
Sequential Manual PID Tuning
      ↓
Baseline Gains Fix
      ↓
Tuning Record Export
```

이 실험에서 정하는 baseline PID는 일부러 나쁜 비교군이 아니라, 이후 `Optimized PID`와 `RL Controller`가 비교할 reasonable conventional controller이다.

In [ ]:
import os, sys, json, platform, subprocess, math
from pathlib import Path


def _in_colab():
    return "google.colab" in sys.modules


def _find_root(start: Path) -> Path:
    p = start.resolve()
    for cand in [p, *p.parents]:
        if (cand / "src").exists() and (cand / "docs").exists():
            return cand
    return p


REPO_URL = "https://github.com/sw030701-ai/motor-control-optimization.git"

if _in_colab():
    root = Path("/content/motor-control-optimization")
    if not root.exists():
        subprocess.run(["git", "clone", REPO_URL, str(root)], check=True)
    ROOT = root
else:
    ROOT = _find_root(Path.cwd())

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

os.environ.setdefault("MPLCONFIGDIR", "/tmp/mplconfig")
os.environ.setdefault("XDG_CACHE_HOME", "/tmp/xdgcache")
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)
Path(os.environ["XDG_CACHE_HOME"]).mkdir(parents=True, exist_ok=True)

import numpy as np
import pandas as pd
import matplotlib
if not _in_colab():
    matplotlib.use("Agg")
import matplotlib.pyplot as plt
plt.rcParams["axes.unicode_minus"] = False


def _git(*args):
    try:
        return subprocess.check_output(["git", *args], cwd=ROOT, text=True).strip()
    except Exception:
        return None

ENV = {
    "root": str(ROOT),
    "python": platform.python_version(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "git_commit": _git("rev-parse", "HEAD"),
    "git_dirty": bool(_git("status", "--short")),
}

print(json.dumps(ENV, indent=2, ensure_ascii=False))

In [ ]:
# ---- Execution switches ----
SAVE_ARTIFACTS = True
SHOW_SCAN_TABLES = True

RESULT_TABLE_DIR = Path("results") / "tables"
RESULT_FIGURE_DIR = Path("results") / "figures"

if SAVE_ARTIFACTS:
    RESULT_TABLE_DIR.mkdir(parents=True, exist_ok=True)
    RESULT_FIGURE_DIR.mkdir(parents=True, exist_ok=True)

print("SAVE_ARTIFACTS:", SAVE_ARTIFACTS)
print("SHOW_SCAN_TABLES:", SHOW_SCAN_TABLES)

## Section 1 — Nominal DC Motor Setup

- **분석 내용**: v1 nominal motor parameter set을 정의하고, `12 V`에서 reachable speed를 계산한다.
- **중요 포인트**: 이 값들은 PID gains가 아니라 plant 자체의 physical parameters이다.

In [ ]:
from src.motor.dc_motor import nominal_dc_motor_params
from src.simulation.pid_simulation import reference_from_reachable_speed

params = nominal_dc_motor_params()
V_MAX = 12.0
REFERENCE_FRACTION = 0.504
OMEGA_REF = round(reference_from_reachable_speed(params, V_MAX, fraction=REFERENCE_FRACTION), 1)
OMEGA_SS_MAX = params.no_load_steady_state_speed(V_MAX)

parameter_table = pd.DataFrame([
    {"Parameter": "R", "Meaning": "Armature resistance", "Value": params.R, "Unit": "ohm"},
    {"Parameter": "L", "Meaning": "Armature inductance", "Value": params.L, "Unit": "H"},
    {"Parameter": "J_m", "Meaning": "Rotor inertia", "Value": params.J_m, "Unit": "kg m^2"},
    {"Parameter": "b", "Meaning": "Viscous friction", "Value": params.b, "Unit": "N m s/rad"},
    {"Parameter": "K_t", "Meaning": "Torque constant", "Value": params.K_t, "Unit": "N m/A"},
    {"Parameter": "K_e", "Meaning": "Back-EMF constant", "Value": params.K_e, "Unit": "V s/rad"},
    {"Parameter": "V_max", "Meaning": "Voltage limit", "Value": V_MAX, "Unit": "V"},
    {"Parameter": "omega_ref", "Meaning": "Reference speed", "Value": OMEGA_REF, "Unit": "rad/s"},
])

display(parameter_table)
print(f"No-load maximum steady-state speed: {OMEGA_SS_MAX:.3f} rad/s")
print(f"Selected reference speed          : {OMEGA_REF:.3f} rad/s")
print(f"Reference / max speed             : {OMEGA_REF / OMEGA_SS_MAX:.3f}")

## Section 2 — Open-Loop Validation

- **분석 내용**: PID controller 없이 constant voltage를 입력했을 때 motor speed가 theoretical steady-state speed에 수렴하는지 확인한다.

Open-loop theoretical steady-state speed는 다음과 같다.

```math
\omega_{ss}
=
\frac{V}{K_e+\frac{Rb}{K_t}}
```

In [ ]:
from src.simulation.pid_simulation import simulate_open_loop

open_loop = simulate_open_loop(
    motor_params=params,
    voltage=V_MAX,
    simulation_time=2.0,
    dt=0.0005,
)

open_loop_final = float(open_loop["omega"][-1])
open_loop_expected = float(params.no_load_steady_state_speed(V_MAX))
open_loop_error = abs(open_loop_final - open_loop_expected)

open_loop_record = pd.DataFrame([{
    "V_test": V_MAX,
    "omega_ss_theory": open_loop_expected,
    "omega_final_sim": open_loop_final,
    "absolute_error": open_loop_error,
    "relative_error_percent": 100 * open_loop_error / open_loop_expected,
}])

display(open_loop_record)

plt.figure(figsize=(9, 4.5))
plt.plot(open_loop["time"], open_loop["omega"], label="Open-loop speed")
plt.axhline(open_loop_expected, linestyle="--", color="tab:red", label="Theoretical steady-state")
plt.xlabel("Time [s]")
plt.ylabel("Angular speed [rad/s]")
plt.title("Open-Loop Step Voltage Response")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()

if SAVE_ARTIFACTS:
    plt.savefig(RESULT_FIGURE_DIR / "open_loop_validation.png", dpi=160)
plt.show()

> **검증 기준**: simulated final speed가 theoretical $\omega_{ss}$와 충분히 가까우면 motor equation, parameter unit, numerical integration이 기본적으로 정상이라고 판단한다.

## Section 3 — Cost Function and Acceptance Criteria

- **분석 내용**: baseline PID를 평가할 cost function과 acceptance criteria를 고정한다.
- **주의**: baseline tuning은 $\mathcal{J}$를 직접 최소화하는 optimization이 아니다. $\mathcal{J}_{baseline}$은 benchmark로만 사용한다.

```math
\mathcal{J}
=
0.60\mathcal{J}_{tracking}
+
0.25\mathcal{J}_{overshoot}
+
0.15\mathcal{J}_{control}
```

In [ ]:
COST_WEIGHTS = {
    "tracking": 0.60,
    "overshoot": 0.25,
    "control": 0.15,
}

ACCEPTANCE = {
    "overshoot_percent_max": 10.0,
    "steady_state_error_percent_max": 2.0,
    "persistent_saturation": "avoid",
    "sustained_oscillation": "avoid",
}

print("Cost weights:")
print(json.dumps(COST_WEIGHTS, indent=2, ensure_ascii=False))
print("\nAcceptance criteria:")
print(json.dumps(ACCEPTANCE, indent=2, ensure_ascii=False))

## Section 4 — Sequential Manual PID Tuning

- **Step 1**: $K_i=0$, $K_d=0$으로 두고 $K_p$를 증가시킨다.
- **Step 2**: $K_p$를 고정하고 $K_i$를 증가시켜 steady-state error를 줄인다.
- **Step 3**: 필요한 경우 작은 $K_d$를 추가해 transient response를 완화한다.

이 scan은 optimizer가 아니라, 사람이 tuning할 때의 판단 규칙을 reproducible하게 기록하기 위한 실험이다.

In [ ]:
from src.simulation.baseline_tuning import BaselineTuningConfig, sequential_baseline_tuning

config = BaselineTuningConfig(
    omega_ref=OMEGA_REF,
    V_max=V_MAX,
    simulation_time=2.0,
    dt=0.0005,
    settling_target=0.5,
)

tuning = sequential_baseline_tuning(params, config)

kp_scan = pd.DataFrame(tuning["Kp_scan"])
ki_scan = pd.DataFrame(tuning["Ki_scan"])
kd_scan = pd.DataFrame(tuning["Kd_scan"])
selected = tuning["selected"]
final_gains = tuning["final_gains"]

columns = [
    "K_p", "K_i", "K_d", "total", "tracking", "overshoot", "control",
    "overshoot_percent", "settling_time", "steady_state_error_percent",
    "voltage_max_abs", "saturation_percent", "omega_final",
]

if SHOW_SCAN_TABLES:
    print("Kp scan")
    display(kp_scan[columns].round(6))
    print("Ki scan")
    display(ki_scan[columns].round(6))
    print("Kd scan")
    display(kd_scan[columns].round(6))

print("Selected baseline gains:")
print(final_gains)

## Section 5 — Baseline Closed-Loop Simulation

- **분석 내용**: 선택된 baseline gains를 고정하고 closed-loop response를 다시 계산한다.
- **결과 해석**: 여기서 계산한 $\mathcal{J}_{baseline}$은 다음 단계의 optimized PID와 비교할 benchmark이다.

In [ ]:
from src.optimization.cost_function import accepted_baseline, compute_cost
from src.simulation.pid_simulation import simulate_pid

baseline_result = simulate_pid(
    motor_params=params,
    gains=final_gains,
    omega_ref=OMEGA_REF,
    V_max=V_MAX,
    simulation_time=config.simulation_time,
    dt=config.dt,
)

baseline_cost = compute_cost(baseline_result, omega_ref=OMEGA_REF, V_max=V_MAX)

baseline_record = pd.DataFrame([{
    "K_p_baseline": final_gains.K_p,
    "K_i_baseline": final_gains.K_i,
    "K_d_baseline": final_gains.K_d,
    "reference_speed_rad_s": OMEGA_REF,
    "V_max": V_MAX,
    "J_baseline": baseline_cost["total"],
    "J_tracking": baseline_cost["tracking"],
    "J_overshoot": baseline_cost["overshoot_cost"],
    "J_control": baseline_cost["control"],
    "overshoot_percent": baseline_cost["overshoot_percent"],
    "steady_state_error_percent": baseline_cost["steady_state_error_percent"],
    "settling_time_s": baseline_cost["settling_time"],
    "max_abs_voltage": baseline_cost["voltage_max_abs"],
    "saturation_percent": baseline_cost["saturation_percent"],
    "accepted_v1": accepted_baseline(baseline_cost),
}])

display(baseline_record.T.rename(columns={0: "value"}))

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 9), sharex=True)

axes[0].plot(baseline_result["time"], baseline_result["omega"], label="Baseline PID speed")
axes[0].axhline(OMEGA_REF, linestyle="--", color="tab:red", label="Reference")
axes[0].set_ylabel("ω [rad/s]")
axes[0].set_title("Baseline PID Closed-Loop Response")
axes[0].grid(True, alpha=0.3)
axes[0].legend()

axes[1].plot(baseline_result["time"], baseline_result["error"], color="tab:orange")
axes[1].axhline(0.0, linestyle="--", color="black", linewidth=1)
axes[1].set_ylabel("e(t) [rad/s]")
axes[1].grid(True, alpha=0.3)

axes[2].plot(baseline_result["time"], baseline_result["voltage"], color="tab:green")
axes[2].axhline(V_MAX, linestyle="--", color="tab:red", linewidth=1)
axes[2].axhline(-V_MAX, linestyle="--", color="tab:red", linewidth=1)
axes[2].set_xlabel("Time [s]")
axes[2].set_ylabel("V(t) [V]")
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
if SAVE_ARTIFACTS:
    plt.savefig(RESULT_FIGURE_DIR / "baseline_pid_response.png", dpi=160)
plt.show()

## Section 6 — Tuning Record Export

아래 cell은 baseline tuning record template을 실제 값으로 채워 저장한다.

생성되는 파일:

```text
results/tables/baseline_pid_tuning_record.csv
results/tables/baseline_pid_tuning_record.md
results/tables/baseline_pid_tuning_record.json
```

In [ ]:
record = baseline_record.iloc[0].to_dict()
record["motor_parameters"] = {
    "R": params.R,
    "L": params.L,
    "J_m": params.J_m,
    "b": params.b,
    "K_t": params.K_t,
    "K_e": params.K_e,
}
record["environment"] = ENV

if SAVE_ARTIFACTS:
    baseline_record.to_csv(RESULT_TABLE_DIR / "baseline_pid_tuning_record.csv", index=False)
    md_rows = ["| Item | Value |", "|---|---:|"]
    for key, value in baseline_record.iloc[0].items():
        if isinstance(value, float):
            text = f"{value:.10g}"
        else:
            text = str(value)
        md_rows.append(f"| `{key}` | {text} |")
    (RESULT_TABLE_DIR / "baseline_pid_tuning_record.md").write_text(
        "\n".join(md_rows) + "\n",
        encoding="utf-8",
    )
    with open(RESULT_TABLE_DIR / "baseline_pid_tuning_record.json", "w", encoding="utf-8") as f:
        json.dump(record, f, indent=2, ensure_ascii=False)

print("Baseline tuning record saved." if SAVE_ARTIFACTS else "SAVE_ARTIFACTS=False, no files saved.")
print(json.dumps({
    "K_p": final_gains.K_p,
    "K_i": final_gains.K_i,
    "K_d": final_gains.K_d,
    "J_baseline": baseline_cost["total"],
    "overshoot_percent": baseline_cost["overshoot_percent"],
    "steady_state_error_percent": baseline_cost["steady_state_error_percent"],
    "settling_time_s": baseline_cost["settling_time"],
    "accepted_v1": accepted_baseline(baseline_cost),
}, indent=2, ensure_ascii=False))

## Final Summary

이번 notebook에서 baseline PID gains는 다음과 같이 고정한다.

```text
K_p_baseline = 0.08
K_i_baseline = 0.80
K_d_baseline = 0.002
```

이 값은 `Manual PID → Random Search → Bayesian Optimization → RL Controller` 비교에서 conventional PID benchmark로 사용한다.

> **주의**: 이 baseline은 intentionally poor baseline이 아니다. 안정적이고 납득 가능한 conventional PID를 기준점으로 세워야 이후 optimization 성능 비교가 왜곡되지 않는다.